# Markdown 标题标准化 — 两层架构逐环节测试

**架构**:
- **第一层**: 找出所有 `#` 候选 → LLM 结合上下文语义判定
- **第二层**: 规则全局重建最终标题等级

下面逐个环节测试，方便调试。

## 0. 环境准备

In [1]:
from pathlib import Path
import json
import os

import local_batch_convert2 as lb2

ROOT = Path.cwd()
SAMPLE = ROOT / "rawdata" / "004_嵌入式系统：硬件与软件架构.md"

assert SAMPLE.exists(), f"找不到: {SAMPLE}"
print("OK! 测试文件:", SAMPLE.name)

OK! 测试文件: 004_嵌入式系统：硬件与软件架构.md


## 1. 提取 # 候选 + 上下文

`extract_headings()` 从 Markdown 中找出所有 `#` 开头行，收集：
- 前后正文片段
- 前后相邻的 # 标题

In [2]:
md_text = SAMPLE.read_text(encoding="utf-8")
headings = lb2.extract_headings(md_text, context_chars=200)

print(f"候选总数: {len(headings)}")
print(f"\n--- 前 5 个候选 ---")
for h in headings[:5]:
    print(f"\n行 {h['line']}: {h['text']}")
    print(f"  前文: {h['prev_text'][:80]}...")
    print(f"  后文: {h['next_text'][:80]}...")
    print(f"  前标题: {h['prev_headings']}")
    print(f"  后标题: {h['next_headings']}")

候选总数: 442

--- 前 5 个候选 ---

行 1: 嵌入式系统
  前文: ...
  后文: # 硬件与软件架构

Embedded Systems Architecture

A Comprehensive Guide for Engineers an...
  前标题: []
  后标题: ['行3: 硬件与软件架构', '行13: 嵌入式系统 硬件与软件架构']

行 3: 硬件与软件架构
  前文: # 嵌入式系统...
  后文: Embedded Systems Architecture

A Comprehensive Guide for Engineers and Programme...
  前标题: ['行1: 嵌入式系统']
  后标题: ['行13: 嵌入式系统 硬件与软件架构', '行28: 本书特色']

行 13: 嵌入式系统 硬件与软件架构
  前文: # 嵌入式系统

# 硬件与软件架构

Embedded Systems Architecture

A Comprehensive Guide for Eng...
  后文: “本书填补了空白。目前的大多数图书都只讲述了嵌入式系统领域的一部分问题，而本书包罗万象，是嵌入式工程师和程序员必备的独一无二的完整指南。我认为它绝对是一部必读之...
  前标题: ['行1: 嵌入式系统', '行3: 硬件与软件架构']
  后标题: ['行28: 本书特色', '行66: Tammy Noergaard']

行 28: 本书特色
  前文: 界的热点。但是，由于涉及技术领域众多，嵌入式系统开发长期以来缺乏比较全面的权威文献。

本书很好地弥补了这一空白。作者是为数不多的从事过嵌入式系统方方面面工作的...
  后文: - 嵌入式系统硬件，包括处理器、存储器、总线和I/O。  
- 嵌入式系统涉及的各种标准，包括程序设计语言、网络等。  
- 嵌入式系统软件，包括设备驱动程序、...
  前标题: ['行3: 硬件与软件架构', '行13: 嵌入式系统 硬件与软件架构']
  后标题: ['行66: Tammy Noergaard', '行83: 嵌入式系统 硬件与软件架构']

行 66: Tammy Noergaard
  前文: ext_image</summary>

## 2. 检查目录区候选

目录区最容易误判，重点看一下

In [3]:
toc_h = [h for h in headings if 70 <= h['line'] <= 120]
for h in toc_h:
    print(f"行 {h['line']}: {h['text']}")
    print(f"  前后标题: {' | '.join(h['prev_headings'])} | {' | '.join(h['next_headings'])}")
    print()

行 83: 嵌入式系统 硬件与软件架构
  前后标题: 行28: 本书特色 | 行66: Tammy Noergaard | 行102: 图书在版编目（CIP）数据 | 行114: 内容提要

行 102: 图书在版编目（CIP）数据
  前后标题: 行66: Tammy Noergaard | 行83: 嵌入式系统 硬件与软件架构 | 行114: 内容提要 | 行120: 图灵计算机科学丛书

行 114: 内容提要
  前后标题: 行83: 嵌入式系统 硬件与软件架构 | 行102: 图书在版编目（CIP）数据 | 行120: 图灵计算机科学丛书 | 行122: 嵌入式系统：硬件与软件架构

行 120: 图灵计算机科学丛书
  前后标题: 行102: 图书在版编目（CIP）数据 | 行114: 内容提要 | 行122: 嵌入式系统：硬件与软件架构 | 行158: 版权声明



## 3. 估算 token 数 + 判断是否需要分批

上下文窗口 16384。如果单次 prompt 超过预算，自动切分。

In [4]:
batches = lb2.batch_headings(headings, SAMPLE.name)
print(f"批数: {len(batches)}")
for i, b in enumerate(batches):
    print(f"  批 {i+1}: {len(b)} 个候选, 行 {b[0]['line']}-{b[-1]['line']}")

# 查看第一批发给 LLM 的 prompt 长度
if len(batches) == 1:
    msgs = lb2.build_messages(batches[0], SAMPLE.name)
else:
    outline = lb2.build_outline(headings)
    msgs = lb2.build_messages(batches[0], SAMPLE.name, outline)

print(f"\nSystem prompt tokens: {lb2.count_tokens(msgs[0]['content'])}")
print(f"User prompt tokens: {lb2.count_tokens(msgs[1]['content'])}")
print(f"总 tokens: {lb2.count_tokens(msgs[0]['content']) + lb2.count_tokens(msgs[1]['content'])}")
print(f"\n--- System Prompt 前 500 字符 ---")
print(msgs[0]['content'][:500])

批数: 25
  批 1: 21 个候选, 行 1-332
  批 2: 18 个候选, 行 362-668
  批 3: 17 个候选, 行 696-1608
  批 4: 19 个候选, 行 1612-2880
  批 5: 16 个候选, 行 2882-3823
  批 6: 20 个候选, 行 4369-5677
  批 7: 20 个候选, 行 5723-7126
  批 8: 16 个候选, 行 7147-8606
  批 9: 23 个候选, 行 8648-9242
  批 10: 18 个候选, 行 9276-10730
  批 11: 18 个候选, 行 10905-11948
  批 12: 16 个候选, 行 12135-13129
  批 13: 18 个候选, 行 13165-14548
  批 14: 20 个候选, 行 14625-15941
  批 15: 18 个候选, 行 15992-17396
  批 16: 18 个候选, 行 17658-18245
  批 17: 19 个候选, 行 18290-20447
  批 18: 15 个候选, 行 20449-21116
  批 19: 16 个候选, 行 21124-21594
  批 20: 16 个候选, 行 21678-22325
  批 21: 16 个候选, 行 22333-22457
  批 22: 15 个候选, 行 22465-22944
  批 23: 18 个候选, 行 22948-23307
  批 24: 21 个候选, 行 23367-23787
  批 25: 10 个候选, 行 23789-24085

System prompt tokens: 412
User prompt tokens: 13126
总 tokens: 13538

--- System Prompt 前 500 字符 ---
你是一个文档标题结构分析师。审查 Markdown 中所有 `#` 候选行，判断真正的正文标题和级别。

## 宏观规则

### H1 (#) — 极其严格
只有这些才是 H1: 论文题目、学位论文题目、书籍书名。其他任何情况都不设 H1。

### H2 (##)
摘要/Abstract、目录(本身)、第X章、Introduction/Method

## 4. 预览一个候选的完整 Prompt 格式

看看发给 LLM 的内容长什么样

In [5]:
sample_h = headings[3:6]  # 只看前 3 个
formatted = "\n\n".join(lb2.format_heading(h) for h in sample_h)
print(formatted[:2000])

--- 行 28 ---
候选行: # 本书特色
前文: 界的热点。但是，由于涉及技术领域众多，嵌入式系统开发长期以来缺乏比较全面的权威文献。

本书很好地弥补了这一空白。作者是为数不多的从事过嵌入式系统方方面面工作的资深专家。在书中，她通过大量案例分析，展现了嵌入式系统的“全景视图”，涵盖硬件层、软件层和系统开发过程，很好地结合了理论与实践。本书可读性强，全面实用，是嵌入式系统工程师的得力助手，同时也非常适合作为高等院校相关专业学生的嵌入式系统教材。
后文: - 嵌入式系统硬件，包括处理器、存储器、总线和I/O。  
- 嵌入式系统涉及的各种标准，包括程序设计语言、网络等。  
- 嵌入式系统软件，包括设备驱动程序、操作系统、中间件和应用软件。  
完整系统的设计和开发全过程。

本书教辅材料在图灵网站上提供下载。

本书译自原版Embedded Systems Architecture: A Comprehensive Guide for Engi
前面标题: 行3: 硬件与软件架构 | 行13: 嵌入式系统 硬件与软件架构
后面标题: 行66: Tammy Noergaard | 行83: 嵌入式系统 硬件与软件架构

--- 行 66 ---
候选行: # Tammy Noergaard
前文: ext_image</summary>

Embedded Systems Architecture
A Comprehensive Guide for Engineers
and Programmers
</details>

Embedded Systems Architecture   
A Comprehensive Guide for Engineers and Programmers
后文: 世界级的嵌入式系统专家，有丰富的嵌入式系统领域开发、设计、营销和培训经验。曾在Sony、Wind River等公司工作，参与或领导开发了众多嵌入式软件和硬件，其中包括被《消费电子产品报告》杂志评为第一的电视机嵌入式系统产品。目前她在Esmertec北美公司担任资深技术专家和顾问，并同时在加州大学伯克利分校和斯坦福大学讲授嵌入式系统课程。

ISBN 978-7-115-16805-4   
![
前面标题: 行13: 嵌入式系统 硬件与软件架构 | 行28: 本书特

## 5. 第一层: 调用 LLM 判定

⚠️ 真实调用 API，会耗时

In [6]:
decisions = lb2.layer1_classify(headings, SAMPLE.name)
print(f"LLM 返回 {len(decisions)} 条判定")
print(f"其中 is_heading=True: {sum(1 for d in decisions if d.get('is_heading'))}")
print(f"其中 is_heading=False: {sum(1 for d in decisions if not d.get('is_heading'))}")

  批次 1/25: 21 候选 行1-332
    第1次尝试失败: 无法解析 JSON，原始文本前500字符:
我需要分析Markdown文档中所有以`#`开头的候选行，判断它们是否是真正的正文标题，以及它们的级别（H1-H6）。

首先，让我回顾一下宏观规则：

### H1 (#) — 极其严格
只有这些才是 H1: 论文题目、学位论文题目、书籍书名。其他任何情况都不设 H1。

### H2 (##)
摘要/Abstract、目录(本身)、第X章、Introduction/Methods/Results/Discussion、参考文献/References、
致谢/Acknowledgements、附录/Appendix、一级数字编号(1./2./3.)

### H3 (###)
1.1/2.3 等二级数字编号、主章节下的分节

### H4 (####)
1.1.1/2.3.4 等三级数字编号

### 非标题
- 目录区条目(只是索引，不是正文标题)
- 封面信息(作者、单位、期刊名)
- 图片说明、表格说明
- 中文学位论文括号编号: (1)/(2)/（一）是段内枚举

### 其他
- 同一标题在目录和正文各出现一次 → 目录的设 is_heading=false, role="
  批次 2/25: 18 候选 行362-668
  批次 3/25: 17 候选 行696-1608
    第1次尝试失败: 无法解析 JSON，原始文本前500字符:
我需要分析提供的Markdown文档中的候选标题行，判断哪些是真正的正文标题，以及它们的级别。我将根据给定的规则来分析每个候选行。

首先，让我回顾一下规则：

### H1 (#) — 极其严格
只有这些才是 H1: 论文题目、学位论文题目、书籍书名。其他任何情况都不设 H1。

### H2 (##)
摘要/Abstract、目录(本身)、第X章、Introduction/Methods/Results/Discussion、参考文献/References、
致谢/Acknowledgements、附录/Appendix、一级数字编号(1./2./3.)

### H3 (###)
1.1/2.3 等二级数字编号、主章节下的分节

### H4 (####)
1.1.1/2.3.4 等三级数

## 6. 查看 LLM 对几个典型位置的判断

In [ ]:
# 看前 15 条
for d in decisions[:15]:
    print(f"行 {d['line']}: h={d.get('is_heading')} lv={d.get('level')} role={d.get('role')} | {d.get('reason','')[:60]}")

## 7. 第二层: 规则重建

将 LLM 结果传入规则引擎，得到最终一致的标题等级

In [7]:
final = lb2.layer2_rebuild(headings, decisions)

headings_only = [d for d in final if d['is_heading']]
non_headings = [d for d in final if not d['is_heading']]

print(f"最终标题: {len(headings_only)} 个")
print(f"最终非标题: {len(non_headings)} 个")
print()

# 按 level 统计
from collections import Counter
level_counts = Counter(d['level'] for d in headings_only)
print("标题级别分布:")
for lv in sorted(level_counts):
    print(f"  H{lv}: {level_counts[lv]} 个")

最终标题: 347 个
最终非标题: 95 个

标题级别分布:
  H1: 1 个
  H2: 127 个
  H3: 141 个
  H4: 76 个
  H5: 2 个


## 8. 对比 LLM 原始判定 vs 规则重建后

In [8]:
changes = 0
for d in final:
    llm_d = next((x for x in decisions if x['line'] == d['line']), {})
    llm_h = llm_d.get('is_heading')
    llm_lv = llm_d.get('level')
    if d['is_heading'] != llm_h or d['level'] != llm_lv:
        changes += 1
        print(f"行 {d['line']}: LLM(h={llm_h},lv={llm_lv}) → 重建(h={d['is_heading']},lv={d['level']}) | {d['reason']}")

print(f"\n共 {changes} 条被规则修正")

行 236: LLM(h=False,lv=None) → 重建(h=True,lv=2) | 致谢
行 248: LLM(h=True,lv=2) → 重建(h=False,lv=None) | 目录条目
行 250: LLM(h=True,lv=2) → 重建(h=False,lv=None) | 目录条目
行 268: LLM(h=True,lv=2) → 重建(h=False,lv=None) | 目录条目
行 294: LLM(h=True,lv=2) → 重建(h=False,lv=None) | 目录条目
行 296: LLM(h=True,lv=2) → 重建(h=False,lv=None) | 目录条目
行 332: LLM(h=True,lv=2) → 重建(h=False,lv=None) | 目录条目
行 362: LLM(h=True,lv=2) → 重建(h=False,lv=None) | 目录条目
行 380: LLM(h=True,lv=2) → 重建(h=False,lv=None) | 目录条目
行 408: LLM(h=True,lv=2) → 重建(h=False,lv=None) | 目录条目
行 426: LLM(h=True,lv=2) → 重建(h=False,lv=None) | 目录条目
行 428: LLM(h=True,lv=2) → 重建(h=False,lv=None) | 目录条目
行 456: LLM(h=True,lv=2) → 重建(h=False,lv=None) | 目录条目
行 488: LLM(h=True,lv=2) → 重建(h=False,lv=None) | 目录条目
行 514: LLM(h=True,lv=2) → 重建(h=False,lv=None) | 目录条目
行 516: LLM(h=True,lv=2) → 重建(h=False,lv=None) | 目录条目
行 538: LLM(h=True,lv=2) → 重建(h=False,lv=None) | 目录条目
行 1753: LLM(h=True,lv=3) → 重建(h=True,lv=2) | 编号 depth=1
行 1835: LLM(h=True,lv=3) → 重建(h=True,lv=2) | 

## 9. 查看最终标题结构

In [9]:
for d in headings_only[:40]:
    indent = "  " * (d['level'] - 1)
    print(f"{indent}行 {d['line']:>5} | {'#' * d['level']} {d['text']}   ({d['role']})")

行    13 | # 嵌入式系统 硬件与软件架构   (book_title)
  行   236 | ## 致谢   (acknowledgements)
  行   246 | ## 目录   (toc_marker)
  行   562 | ## 第一部分 嵌入式系统导论   (chapter)
  行   566 | ## 嵌入式系统设计的系统工程方法   (section)
    行   568 | ### 本章内容提要   (section)
    行   576 | ### 1.1 什么是嵌入式系统   (subsection)
    行   594 | ### 1.2 嵌入式系统设计   (subsection)
    行   637 | ### 1.3 嵌入式系统体系结构简介   (subsection)
    行   648 | ### 1.4 嵌入式系统体系结构的重要性   (subsection)
    行   668 | ### 1.5 嵌入式系统模型   (subsection)
  行   779 | ## 了解标准   (section)
    行   862 | ### 2.1 程序设计语言概述和程序设计语言标准实例   (subsection)
      行  1265 | #### 2.1.1 垃圾收集   (subsubsection)
      行  1424 | #### 2.1.2 处理Java字节码   (subsubsection)
    行  1567 | ### 2.2 标准与连网   (subsection)
      行  1608 | #### 2.2.1 相连的设备间的距离   (subsubsection)
      行  1616 | #### 2.2.2 物理介质   (subsubsection)
      行  1640 | #### 2.2.3 网络的体系结构   (subsubsection)
      行  1652 | #### 2.2.4 开放系统互连模型   (subsubsection)
  行  1753 | ## 1. OSI模型与实际的协议栈   (section)
  行  1835 | ## 2. OSI模型第1层：物理层   (sectio

## 10. 应用判定 → 输出标准化 MD

In [10]:
fixed_text = lb2.apply_decisions(md_text, final)

out_path = ROOT / "standardized_md2" / SAMPLE.name
out_path.parent.mkdir(parents=True, exist_ok=True)
out_path.write_text(fixed_text, encoding="utf-8")
print("输出:", out_path)
print(f"原始行数: {len(md_text.split(chr(10)))}")
print(f"修复行数: {len(fixed_text.split(chr(10)))}")

输出: e:\DataDesktop\StudyNotes\PythonLLM学习\KG-RAG-Study\project2_TittleStandardlized\standardized_md2\004_嵌入式系统：硬件与软件架构.md
原始行数: 24099
修复行数: 24099


## 11. 预览修复后的标题 (前 40 条)

In [ ]:
lines = fixed_text.split(chr(10))
count = 0
for i, line in enumerate(lines):
    if line.strip().startswith('#'):
        print(f"行 {i+1:>5}: {line.strip()}")
        count += 1
        if count >= 40:
            break

行    13: # 嵌入式系统 硬件与软件架构
行   176: #08-01 Winsland House I
行   236: ## 致谢
行   246: ## 目录
行   562: ## 第一部分 嵌入式系统导论
行   566: ## 嵌入式系统设计的系统工程方法
行   568: ### 本章内容提要
行   576: ### 1.1 什么是嵌入式系统
行   594: ### 1.2 嵌入式系统设计
行   637: ### 1.3 嵌入式系统体系结构简介
行   648: ### 1.4 嵌入式系统体系结构的重要性
行   668: ### 1.5 嵌入式系统模型
行   779: ## 了解标准
行   862: ### 2.1 程序设计语言概述和程序设计语言标准实例
行  1265: #### 2.1.1 垃圾收集
行  1424: #### 2.1.2 处理Java字节码
行  1567: ### 2.2 标准与连网
行  1608: #### 2.2.1 相连的设备间的距离
行  1616: #### 2.2.2 物理介质
行  1640: #### 2.2.3 网络的体系结构
行  1652: #### 2.2.4 开放系统互连模型
行  1753: ## 1. OSI模型与实际的协议栈
行  1835: ## 2. OSI模型第1层：物理层
行  1997: ## 3. OSI模型第2层：数据链路层
行  2348: ## 4. OSI模型第3层：网络层
行  2437: ## 5. OSI模型第4层：传输层
行  2518: ## 6. OSI模型第5层：会话层
行  2582: ## 7. OSI模型第6层：表示层
行  2636: ## 8. OSI模型第7层：应用层
行  2671: ### 2.3 基于多个标准的设备实例：数字电视
行  2750: ## 小结
行  2756: ## 习题
行  2839: ## 附注
行  2874: ## 第二部分 嵌入式硬件
行  2880: ## 嵌入式硬件构建模块和嵌入式电路板
行  2882: ### 本章内容提要
行  2889: ### 3.1 硬件第一课：学习阅读电路原理图
行  3093: ### 3.2 嵌入式电路板和冯·诺依曼模型
行  3316: ### 3.3 硬

## 🔍 诊断: 查看目录范围和标题丢失原因

看看第二层规则引擎内部做了什么

In [ ]:
# 1. 看 _toc_ranges 检测到了什么
import re

print("=== 检测到的目录范围 ===")
ranges = lb2._toc_ranges(headings)
if ranges:
    for r in ranges:
        print(f"  目录标记: 行 {r['start']} → 目录尾: 行 {r['end']}")
        print(f"  覆盖: {r['end'] - r['start']} 行")
        in_range = [h for h in headings if r['start'] <= h['line'] <= r['end']]
        print(f"  包含 {len(in_range)} 个候选标题")
        print(f"  其中前10个: {[h['text'][:50] for h in in_range[:10]]}")
else:
    print("  未检测到目录范围！")

# 2. 看 clean_title 的效果
print("\n=== 标题清理效果 ===")
sample_texts = [
    "1.1 研究背景.....................15",
    "2.3.4 实验方法......42",
    "第一章 绪论",
    "参考文献",
]
for t in sample_texts:
    print(f"  '{t}' → '{lb2.clean_title(t)}'")

# 3. 数被标记为 toc_entry / paragraph 的标题
toc_count = sum(1 for d in final if d['role'] == 'toc_entry')
para_count = sum(1 for d in final if d['role'] == 'paragraph')
print(f"\n=== 非标题统计 ===")
print(f"  toc_entry (目录条目): {toc_count} 个")
print(f"  paragraph (段内文本): {para_count} 个")

# 4. 如果有很多 toc_entry，列出前 30 个
if toc_count > 5:
    print(f"\n=== 被标记为 toc_entry 的标题 (前30) ===")
    for d in final:
        if d['role'] == 'toc_entry':
            print(f"  行 {d['line']:>5}: {d['text'][:60]}")
        if sum(1 for x in final if x['role'] == 'toc_entry' and x['line'] <= d['line']) >= 30:
            break

## 12. 批量转换文件夹内所有 md 文件

设置好输入文件夹路径，逐个处理。支持缓存断点续跑。

In [ ]:
# ===== 批量转换文件夹 (独立 cell，不依赖前面 cell) =====
INPUT_DIR = r"E:\DataDesktop\StudyNotes\PythonLLM学习\KG-RAG-Study\project2_TittleStandardlized\rawdata"
CACHE_DIR = r".cache_batch"          # 批缓存目录，支持断点续跑

# 导入所有需要的库
from pathlib import Path
import time
import local_batch_convert2 as lb2

# 扫描 md 文件
files = list(Path(INPUT_DIR).glob("*.md"))
if not files:
    files = list(Path(INPUT_DIR).rglob("*.md"))
print(f"找到 {len(files)} 个文件\n")

results = []
for i, f in enumerate(files):
    t0 = time.time()
    out = str(Path("standardized_md2") / f.name)
    js = str(Path("heading_json2") / f"{f.stem}.headings.json")

    # 断点续跑: 跳过已完成的
    if Path(out).exists() and Path(js).exists():
        print(f"[{i+1}/{len(files)}] 跳过 (已完成): {f.name}")
        continue

    print(f"\n{'='*60}")
    print(f"[{i+1}/{len(files)}] {f.name}")
    print(f"{'='*60}")
    r = lb2.process_file(str(f), output_path=out, json_path=js, cache_dir=CACHE_DIR)
    r["elapsed"] = time.time() - t0
    results.append(r)
    print(f"  耗时: {r['elapsed']:.0f}s")

print(f"\n完成 {len(results)} 个文件")
total = sum(r.get("elapsed", 0) for r in results)
print(f"总耗时: {total:.0f}s = {total/60:.1f}分钟")